# 09 — Semantic Document Search / Anlamsal Doküman Arama

**TR:** Bu projede dokümanları parçalara ayırıp embedding vektörlerine dönüştürüyor, kullanıcı sorgusuna anlamca en yakın parçaları cosine similarity ile buluyoruz.

**EN:** In this project, we split documents into chunks, convert them into embedding vectors, and retrieve the chunks that are semantically closest to the user's query using cosine similarity.

**RAG bağlantısı / RAG connection:**  
`Documents → Chunks → Embeddings → Query → Similarity → Top-K Relevant Chunks`


## 1. Kurulum / Installation

**TR:** `sentence-transformers` kütüphanesini kuruyoruz. Bu kütüphane metinleri embedding vektörlerine çevirmek için kullanılır.

**EN:** We install `sentence-transformers`, which is used to convert text into embedding vectors.


In [1]:
!pip install -q -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 3.3 MB/s eta 0:00:00


## 2. Kütüphaneler / Imports

**TR:** `SentenceTransformer` embedding modeli yükler. `cos_sim` iki vektör arasındaki cosine similarity değerini hesaplar.

**EN:** `SentenceTransformer` loads the embedding model. `cos_sim` calculates cosine similarity between vectors.


In [2]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

## 3. Embedding Modeli / Embedding Model

**TR:** Türkçe dahil çok dilli metinleri 384 boyutlu vektörlere dönüştüren modeli yüklüyoruz.

**EN:** We load a multilingual model that converts text, including Turkish, into 384-dimensional vectors.


In [3]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

model = SentenceTransformer(MODEL_NAME)

print("Model yüklendi:")
print(MODEL_NAME)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model yüklendi:
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


## 4. Doküman Havuzu / Document Collection

**TR:** Motor bakımı, PLC ve hidrolik hakkında küçük bir doküman havuzu oluşturuyoruz. Gerçek bir RAG sisteminde bunlar PDF, teknik el kitabı veya veri tabanı içeriği olabilir.

**EN:** We create a small document collection about motor maintenance, PLCs, and hydraulics. In a real RAG system, these could come from PDFs, manuals, or databases.


In [4]:
documents = {

    "motor_bakim": """
Elektrik motorlarında rulmanlar rotorun düzgün ve düşük
sürtünmeyle dönmesini sağlar.

Rulmanlarda yetersiz yağlama, yanlış hizalama, kirlenme
ve aşırı yük aşınmaya neden olabilir.

Rulman arızaları motor titreşiminde ve çalışma sıcaklığında
artışa yol açabilir.

Titreşim analizi, sıcaklık ölçümü ve akustik izleme
kestirimci bakım amacıyla kullanılabilir.
""",

    "plc": """
PLC sistemleri endüstriyel otomasyonda makinelerin
kontrol edilmesi için kullanılır.

Dijital giriş modülleri buton, sensör ve limit switch gibi
elemanlardan gelen sinyalleri işler.

Dijital çıkış modülleri kontaktör, röle ve lambalar gibi
elemanları kontrol edebilir.

PLC programlarında girişler okunur, kontrol mantığı
çalıştırılır ve çıkışlar güncellenir.
""",

    "hidrolik": """
Hidrolik sistemlerde basınçlı sıvı kullanılarak mekanik
hareket elde edilir.

Hidrolik pompalar mekanik enerjiyi hidrolik enerjiye
dönüştürür.

Basınç düşmesi kaçak, pompa aşınması veya valf arızası
gibi nedenlerden kaynaklanabilir.

Filtrelerin düzenli değiştirilmesi hidrolik sistemlerde
kirlenmeden kaynaklanan arızaları azaltır.
"""
}

## 5. Chunking

**TR:** Uzun dokümanları küçük parçalara ayırıyoruz. Her parçaya kaynak doküman adını metadata olarak ekliyoruz.

**EN:** We split long documents into smaller chunks and keep the source document name as metadata for each chunk.


In [5]:
def create_chunks(documents):

    chunks = []

    for document_name, text in documents.items():

        paragraphs = text.strip().split("\n\n")

        for paragraph in paragraphs:

            clean_paragraph = paragraph.replace(
                "\n",
                " "
            ).strip()

            if clean_paragraph:

                chunks.append(
                    {
                        "document": document_name,
                        "text": clean_paragraph
                    }
                )

    return chunks

## 6. Chunk'ları Oluşturma / Creating Chunks

**TR:** `create_chunks()` fonksiyonunu çalıştırarak toplam kaç bilgi parçası elde ettiğimizi görüyoruz.

**EN:** We run `create_chunks()` and check how many information chunks were created.


In [6]:
chunks = create_chunks(documents)

print("Toplam chunk:", len(chunks))

Toplam chunk: 12


## 7. Chunk Kontrolü / Inspecting Chunks

**TR:** Her chunk'ın hangi dokümana ait olduğunu ve içeriğini ekrana yazdırıyoruz.

**EN:** We print each chunk together with its source document.


In [7]:
for chunk in chunks:

    print(
        chunk["document"],
        "→",
        chunk["text"]
    )

motor_bakim → Elektrik motorlarında rulmanlar rotorun düzgün ve düşük sürtünmeyle dönmesini sağlar.
motor_bakim → Rulmanlarda yetersiz yağlama, yanlış hizalama, kirlenme ve aşırı yük aşınmaya neden olabilir.
motor_bakim → Rulman arızaları motor titreşiminde ve çalışma sıcaklığında artışa yol açabilir.
motor_bakim → Titreşim analizi, sıcaklık ölçümü ve akustik izleme kestirimci bakım amacıyla kullanılabilir.
plc → PLC sistemleri endüstriyel otomasyonda makinelerin kontrol edilmesi için kullanılır.
plc → Dijital giriş modülleri buton, sensör ve limit switch gibi elemanlardan gelen sinyalleri işler.
plc → Dijital çıkış modülleri kontaktör, röle ve lambalar gibi elemanları kontrol edebilir.
plc → PLC programlarında girişler okunur, kontrol mantığı çalıştırılır ve çıkışlar güncellenir.
hidrolik → Hidrolik sistemlerde basınçlı sıvı kullanılarak mekanik hareket elde edilir.
hidrolik → Hidrolik pompalar mekanik enerjiyi hidrolik enerjiye dönüştürür.
hidrolik → Basınç düşmesi kaçak, pompa aşınm

## 8. Sadece Metinleri Alma / Extracting Chunk Texts

**TR:** Embedding modeline göndermek için chunk sözlüklerinden yalnızca metin alanlarını çıkarıyoruz.

**EN:** We extract only the text fields from the chunk dictionaries before sending them to the embedding model.


In [8]:
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

## 9. Chunk Embeddingleri / Chunk Embeddings

**TR:** Her chunk'ı sayısal bir embedding vektörüne dönüştürüyoruz. Bu işlem dokümanların anlamsal temsilini oluşturur.

**EN:** We convert every chunk into a numerical embedding vector, creating a semantic representation of the document content.


In [9]:
chunk_embeddings = model.encode(
    chunk_texts
)

## 10. Embedding Boyutu / Embedding Shape

**TR:** Matris boyutunu kontrol ediyoruz. Örneğin `(12, 384)`, 12 chunk ve her chunk için 384 sayı demektir.

**EN:** We inspect the matrix shape. For example, `(12, 384)` means 12 chunks with 384 numbers per chunk.


In [10]:
print(
    chunk_embeddings.shape
)

(12, 384)


## 11. Kullanıcı Sorgusu / User Query

**TR:** Arama yapmak istediğimiz doğal dil sorusunu tanımlıyoruz.

**EN:** We define the natural-language query that we want to search for.


In [11]:
query = "Motor neden aşırı titreşim yapabilir?"

## 12. Sorgu Embeddingi / Query Embedding

**TR:** Kullanıcı sorusunu da dokümanlarla aynı embedding uzayına dönüştürüyoruz.

**EN:** We encode the user query into the same embedding space as the document chunks.


In [12]:
query_embedding = model.encode(
    query
)

## 13. Cosine Similarity

**TR:** Sorgu vektörünü bütün chunk vektörleriyle karşılaştırıyoruz. Yüksek skor genellikle daha güçlü anlamsal benzerlik anlamına gelir.

**EN:** We compare the query vector with all chunk vectors. A higher score generally indicates stronger semantic similarity.


In [13]:
scores = cos_sim(
    query_embedding,
    chunk_embeddings
)[0]

## 14. Ham Skorları Görme / Inspecting Raw Scores

**TR:** Her chunk'ın sorguyla benzerlik skorunu görüntülüyoruz.

**EN:** We display the similarity score of each chunk against the query.


In [14]:
for chunk, score in zip(
    chunks,
    scores
):

    print(
        round(score.item(), 4),
        "→",
        chunk["text"]
    )

0.5375 → Elektrik motorlarında rulmanlar rotorun düzgün ve düşük sürtünmeyle dönmesini sağlar.
0.3985 → Rulmanlarda yetersiz yağlama, yanlış hizalama, kirlenme ve aşırı yük aşınmaya neden olabilir.
0.6608 → Rulman arızaları motor titreşiminde ve çalışma sıcaklığında artışa yol açabilir.
0.2454 → Titreşim analizi, sıcaklık ölçümü ve akustik izleme kestirimci bakım amacıyla kullanılabilir.
0.2297 → PLC sistemleri endüstriyel otomasyonda makinelerin kontrol edilmesi için kullanılır.
0.1707 → Dijital giriş modülleri buton, sensör ve limit switch gibi elemanlardan gelen sinyalleri işler.
0.1492 → Dijital çıkış modülleri kontaktör, röle ve lambalar gibi elemanları kontrol edebilir.
0.0186 → PLC programlarında girişler okunur, kontrol mantığı çalıştırılır ve çıkışlar güncellenir.
0.3661 → Hidrolik sistemlerde basınçlı sıvı kullanılarak mekanik hareket elde edilir.
0.2663 → Hidrolik pompalar mekanik enerjiyi hidrolik enerjiye dönüştürür.
0.5204 → Basınç düşmesi kaçak, pompa aşınması veya valf 

## 15. Semantic Search Fonksiyonu / Semantic Search Function

**TR:** Sorguyu embed eden, bütün chunk'larla karşılaştıran, sonuçları skora göre sıralayan ve en iyi `top_k` sonucu döndüren yeniden kullanılabilir bir arama fonksiyonu oluşturuyoruz.

**EN:** We build a reusable search function that embeds the query, compares it with all chunks, ranks the results, and returns the best `top_k` matches.


In [15]:
def semantic_document_search(
    query,
    chunks,
    chunk_embeddings,
    model,
    top_k=3
):

    query_embedding = model.encode(
        query
    )

    scores = cos_sim(
        query_embedding,
        chunk_embeddings
    )[0]

    results = []

    for index, score in enumerate(scores):

        results.append(
            {
                "document": chunks[index]["document"],
                "text": chunks[index]["text"],
                "score": score.item()
            }
        )

    results = sorted(
        results,
        key=lambda x: x["score"],
        reverse=True
    )

    return results[:top_k]

## 16. Motor Sorgusu / Motor Query

**TR:** Motor titreşimiyle ilgili sorgu için en alakalı üç chunk'ı getiriyoruz.

**EN:** We retrieve the three most relevant chunks for a motor-vibration query.


In [16]:
results = semantic_document_search(
    query="Motor neden aşırı titreşim yapabilir?",
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=model,
    top_k=3
)

## 17. Sonuçları Sıralı Gösterme / Ranked Results

**TR:** Sonuçların sırasını, kaynak dokümanı, benzerlik skorunu ve metni okunabilir biçimde gösteriyoruz.

**EN:** We display the rank, source document, similarity score, and text in a readable format.


In [17]:
for rank, result in enumerate(
    results,
    start=1
):

    print("=" * 70)

    print(
        "Sıra:",
        rank
    )

    print(
        "Doküman:",
        result["document"]
    )

    print(
        "Benzerlik:",
        round(
            result["score"],
            4
        )
    )

    print(
        "Metin:",
        result["text"]
    )

Sıra: 1
Doküman: motor_bakim
Benzerlik: 0.6608
Metin: Rulman arızaları motor titreşiminde ve çalışma sıcaklığında artışa yol açabilir.
Sıra: 2
Doküman: motor_bakim
Benzerlik: 0.5375
Metin: Elektrik motorlarında rulmanlar rotorun düzgün ve düşük sürtünmeyle dönmesini sağlar.
Sıra: 3
Doküman: hidrolik
Benzerlik: 0.5204
Metin: Basınç düşmesi kaçak, pompa aşınması veya valf arızası gibi nedenlerden kaynaklanabilir.


## 18. PLC Sorgusu / PLC Query

**TR:** Aynı retrieval sistemini PLC sensör girişleri hakkında farklı bir soruyla test ediyoruz.

**EN:** We test the same retrieval system with a different question about PLC sensor inputs.


In [18]:
results = semantic_document_search(
    query="Sensörlerden gelen sinyalleri PLC nasıl alır?",
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=model,
    top_k=3
)

for result in results:

    print(
        round(result["score"], 4),
        "→",
        result["document"],
        "→",
        result["text"]
    )

0.5471 → plc → Dijital giriş modülleri buton, sensör ve limit switch gibi elemanlardan gelen sinyalleri işler.
0.4784 → plc → PLC programlarında girişler okunur, kontrol mantığı çalıştırılır ve çıkışlar güncellenir.
0.4641 → plc → Dijital çıkış modülleri kontaktör, röle ve lambalar gibi elemanları kontrol edebilir.


## 19. Hidrolik Sorgusu / Hydraulic Query

**TR:** Arama sisteminin başka bir teknik alanda da doğru chunk'ı bulup bulmadığını test ediyoruz.

**EN:** We test whether the search system can retrieve the correct chunk for another technical domain.


In [19]:
results = semantic_document_search(
    query="Hidrolik sistemde basınç neden düşer?",
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=model,
    top_k=3
)

for result in results:

    print(
        round(result["score"], 4),
        "→",
        result["document"],
        "→",
        result["text"]
    )

0.7832 → hidrolik → Hidrolik sistemlerde basınçlı sıvı kullanılarak mekanik hareket elde edilir.
0.6308 → hidrolik → Hidrolik pompalar mekanik enerjiyi hidrolik enerjiye dönüştürür.
0.5489 → hidrolik → Basınç düşmesi kaçak, pompa aşınması veya valf arızası gibi nedenlerden kaynaklanabilir.


## Sonuç / Conclusion

**TR:** Bu notebook küçük bir semantic retriever oluşturur. Sistem cevap üretmez; yalnızca sorguya anlamca en yakın doküman parçalarını bulur. Bir sonraki adımda bu retrieved chunk'ları bir LLM'e vererek RAG oluşturabiliriz.

**EN:** This notebook builds a small semantic retriever. It does not generate answers; it only finds document chunks that are semantically closest to the query. The next step is to pass those retrieved chunks to an LLM to build a RAG pipeline.

### Öğrenilen ana kavramlar / Key concepts
- Chunking
- Embeddings
- Cosine similarity
- Semantic search
- Top-K retrieval
- Metadata
- Retriever
